# ICU Mortality Prediction using Machine Learning

This notebook builds a predictive model for in-hospital mortality in ICU patients using clinical data.

We apply:
- Data preprocessing
- Machine learning models
- Evaluation metrics
- Explainable AI (SHAP)


##Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

import shap


## Load Dataset

In [ ]:
df = pd.read_csv("data01.csv", index_col=0)
df.head()

## Class Distribution Plot

In [ ]:
df['outcome'].value_counts().plot(kind='bar')
plt.title("Outcome distribution")
plt.show()

## Missing Values Handling

In [ ]:
features = df.iloc[:, 2:]
labels = df['outcome']

imputer = KNNImputer(n_neighbors=5)
X_imputed = imputer.fit_transform(features)


## Scaling

In [ ]:

scaler = StandardScaler()
X = scaler.fit_transform(X_imputed)
y = labels.values

## Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

## Decision Tree Model

In [ ]:
dt = DecisionTreeClassifier(class_weight='balanced', random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

print(classification_report(y_test, y_pred_dt))

## Random Forest Model

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print(classification_report(y_test, y_pred_rf))

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)

sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

## ROC Curve

In [ ]:
y_prob = rf.predict_proba(X_test)[:,1]

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0,1],[0,1],'--')
plt.legend()
plt.title("ROC Curve")
plt.show()

## SHAP Explainability

In [ ]:
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values[1], X_test)

## Conclusion

- Random Forest performed best overall
- Class imbalance significantly impacts prediction
- Accuracy alone is misleading in medical datasets
- SHAP improves interpretability of clinical predictions

This model demonstrates how machine learning can support ICU risk stratification.